# 0 air temperature find out the latest 30 years

In [ ]:
# https://weatherspark.com/y/127039/Average-Weather-in-Xiantao-China-Year-Round

In [31]:
work_space= r'D:\shanghai workshop\lst\original'

# 01 landsat data process and HR

In [32]:
import glob
import os

tif_files = glob.glob(os.path.join(work_space,r'\*.tif'))

for file_path in tif_files:
    print("📂 读取 TIF 文件:", file_path)
    # 这里写读取代码


# 删除黑色区域的数据

In [33]:
import glob

tif_files = glob.glob(os.path.join(work_space,r'*QA_PIXEL.tif'))

for file_path in tif_files:
    print("📂 读取 TIF 文件:", file_path)
    # 这里写读取代码

In [34]:
import os
import glob
import rasterio
import numpy as np

tif_files = glob.glob(os.path.join(work_space,r'*QA_PIXEL.tif'))

for file_path in tif_files:
    with rasterio.open(file_path) as src:
        data = src.read(1)  # 读取第一个波段
        profile = src.profile

    # 设置 nodata 值
    nodata_val = 0
    profile['nodata'] = nodata_val

    # 将值为 1 的像素设为 nodata
    data_cleaned = np.where(data == 1, nodata_val, data)

    # 输出路径，添加 `_clean` 后缀
    dirname, basename = os.path.split(file_path)
    name, ext = os.path.splitext(basename)
    out_folder = os.path.join(dirname,r'clean')
    os.makedirs(out_folder,exist_ok=True)
    out_path = os.path.join(out_folder, f"{name}_clean{ext}")

    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(data_cleaned, 1)

    print(f"✅ 已保存清理后的文件: {out_path}")


# merge

In [36]:
import os
import glob
import rasterio
from rasterio.merge import merge
import re

def extract_date_from_filename(filename):
    """从文件名中提取第 4 个下划线后的 8 位日期"""
    parts = os.path.basename(filename).split('_')
    if len(parts) > 4 and re.match(r'\d{8}', parts[3]):
        return parts[3]
    return None

def get_all_dates(folder, suffix="_clean.TIF"):
    tif_files = glob.glob(os.path.join(folder, f"*{suffix}"))
    date_set = set()
    for file in tif_files:
        parts = os.path.basename(file).split('_')
        if len(parts) > 4 and re.match(r'\d{8}', parts[3]):
            date_set.add(parts[3])  # 确保只取第 4 个字段
    return sorted(list(date_set))

def merge_clean_tifs_by_date(folder, date_str, output_name_prefix="LC08_merged", suffix="_clean.TIF"):
    matching_files = []
    tif_files = glob.glob(os.path.join(folder, f"*{suffix}"))

    for f in tif_files:
        parts = os.path.basename(f).split('_')
        if len(parts) > 4 and parts[3] == date_str:
            matching_files.append(f)

    if not matching_files:
        print(f"❌ 没有找到严格匹配日期 {date_str} 的文件")
        return

    print(f"\n📅 合并日期：{date_str}  ▶️ 文件数：{len(matching_files)}")
    for f in matching_files:
        print("   📄", os.path.basename(f))

    src_files = [rasterio.open(f) for f in matching_files]
    mosaic, out_trans = merge(src_files)

    out_meta = src_files[0].meta.copy()
    out_meta.update({
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_trans,
        "nodata": src_files[0].nodata,
        "driver": "GTiff"
    })

    output_folder = os.path.join(folder, 'merge')
    os.makedirs(output_folder, exist_ok=True)

    output_path = os.path.join(output_folder, f"{output_name_prefix}_{date_str}{suffix}")
    with rasterio.open(output_path, "w", **out_meta) as dst:
        dst.write(mosaic)

    print(f"✅ 合并完成: {output_path}")


def merge_all_dates(folder, suffix="_clean.TIF"):
    all_dates = get_all_dates(folder, suffix)
    if not all_dates:
        print("⚠️ 没有找到任何带日期的匹配文件")
        return

    print(f"🔎 共发现 {len(all_dates)} 个日期: {all_dates}")
    for date_str in all_dates:
        merge_clean_tifs_by_date(folder, date_str, suffix=suffix)


merge_all_dates(os.path.join(work_space, 'clean'), suffix="_QA_PIXEL_clean.TIF")


🔎 共发现 144 个日期: ['20150702', '20150709', '20150718', '20150725', '20150803', '20150810', '20150819', '20150826', '20160602', '20160609', '20160618', '20160625', '20160704', '20160711', '20160720', '20160727', '20160805', '20160812', '20160821', '20160828', '20170605', '20170612', '20170621', '20170628', '20170707', '20170714', '20170723', '20170730', '20170808', '20170815', '20170824', '20170831', '20180608', '20180615', '20180624', '20180701', '20180710', '20180717', '20180726', '20180802', '20180811', '20180818', '20180827', '20190602', '20190611', '20190618', '20190627', '20190704', '20190713', '20190720', '20190729', '20190805', '20190814', '20190821', '20190830', '20200604', '20200613', '20200620', '20200629', '20200706', '20200715', '20200722', '20200731', '20200807', '20200816', '20200823', '20210607', '20210616', '20210623', '20210702', '20210709', '20210718', '20210725', '20210803', '20210810', '20210819', '20210826', '20220602', '20220603', '20220610', '20220611', '20220618', 

In [38]:
import os
import glob
import rasterio
from rasterio.merge import merge

def extract_date_from_filename(filename):
    parts = os.path.basename(filename).split('_')
    if len(parts) > 4:
        return parts[3]  # 第 4 个字段是日期
    return None

def get_all_dates(folder, suffix="_ST_B10.TIF"):
    tif_files = glob.glob(os.path.join(folder, f"*{suffix}"))
    date_set = set()
    for file in tif_files:
        date_str = extract_date_from_filename(file)
        if date_str:
            date_set.add(date_str)
    return sorted(list(date_set))

def merge_clean_tifs_by_date(folder, date_str, output_name_prefix="LC08_merged", suffix="_ST_B10.TIF"):
    matching_files = []
    tif_files = glob.glob(os.path.join(folder, f"*{suffix}"))

    for f in tif_files:
        parts = os.path.basename(f).split('_')
        if len(parts) > 4 and parts[3] == date_str:
            matching_files.append(f)

    if not matching_files:
        print(f"❌ 没有找到匹配日期 {date_str} 的文件")
        return

    print(f"\n📅 合并日期：{date_str}  ▶️ 文件数：{len(matching_files)}")
    for f in matching_files:
        print("   📄", os.path.basename(f))

    src_files = [rasterio.open(f) for f in matching_files]
    mosaic, out_trans = merge(src_files)

    out_meta = src_files[0].meta.copy()
    out_meta.update({
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_trans,
        "nodata": src_files[0].nodata,
        "driver": "GTiff"
    })

    output_folder = os.path.join(folder,'clean', 'merge')
    os.makedirs(output_folder, exist_ok=True)

    # 安全提取 band 名（比如 "_ST_B10.TIF" -> "ST_B10"）
    band_part = suffix.strip("_").replace(".TIF", "")
    output_filename = f"{output_name_prefix}_{date_str}_{band_part}.TIF"

    output_path = os.path.join(output_folder, output_filename)

    with rasterio.open(output_path, "w", **out_meta) as dst:
        dst.write(mosaic)

    print(f"✅ 合并完成: {output_path}")

def merge_all_dates(folder, suffix="_ST_B10.TIF"):
    all_dates = get_all_dates(folder, suffix)
    if not all_dates:
        print("⚠️ 没有找到任何带日期的文件")
        return

    print(f"🔎 共发现 {len(all_dates)} 个日期: {all_dates}")
    for date_str in all_dates:
        merge_clean_tifs_by_date(folder, date_str, suffix=suffix)

# ✅ 调用
merge_all_dates(work_space, suffix="_ST_B10.TIF")


🔎 共发现 144 个日期: ['20150702', '20150709', '20150718', '20150725', '20150803', '20150810', '20150819', '20150826', '20160602', '20160609', '20160618', '20160625', '20160704', '20160711', '20160720', '20160727', '20160805', '20160812', '20160821', '20160828', '20170605', '20170612', '20170621', '20170628', '20170707', '20170714', '20170723', '20170730', '20170808', '20170815', '20170824', '20170831', '20180608', '20180615', '20180624', '20180701', '20180710', '20180717', '20180726', '20180802', '20180811', '20180818', '20180827', '20190602', '20190611', '20190618', '20190627', '20190704', '20190713', '20190720', '20190729', '20190805', '20190814', '20190821', '20190830', '20200604', '20200613', '20200620', '20200629', '20200706', '20200715', '20200722', '20200731', '20200807', '20200816', '20200823', '20210607', '20210616', '20210623', '20210702', '20210709', '20210718', '20210725', '20210803', '20210810', '20210819', '20210826', '20220602', '20220603', '20220610', '20220611', '20220618', 

# clip by the boundary

In [45]:
import os
import rasterio
import geopandas as gpd
from rasterio.mask import mask
from shapely.geometry import box
from glob import glob

def clip_rasters_with_shapefile(raster_folder, shapefile_path, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    # Load the shapefile as a GeoDataFrame
    polygon = gpd.read_file(shapefile_path)

    for raster_path in glob(os.path.join(raster_folder, '*.tif')):
        with rasterio.open(raster_path) as src:
            raster_crs = src.crs

            # Reproject polygon to raster CRS if needed
            if polygon.crs != raster_crs:
                polygon_proj = polygon.to_crs(raster_crs)
            else:
                polygon_proj = polygon

            # Raster bounds as a Shapely box
            raster_bounds = box(*src.bounds)

            # Polygon total bounds (merged geometry)
            polygon_union = polygon_proj.unary_union
            polygon_bounds = polygon_union.bounds
            polygon_box = box(*polygon_bounds)

            # Only clip if raster fully contains the polygon's bounding box
            if not raster_bounds.contains(polygon_box):
                print(f"❌ 跳过（raster 不包含 polygon）: {os.path.basename(raster_path)}")
                continue

            # Clip raster
            out_image, out_transform = mask(src, polygon_proj.geometry, crop=True)

            # Update metadata
            out_meta = src.meta.copy()
            out_meta.update({
                "driver": "GTiff",
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "crs": raster_crs
            })

            output_raster_path = os.path.join(output_folder, f"clipped_{os.path.basename(raster_path)}")
            with rasterio.open(output_raster_path, "w", **out_meta) as dest:
                dest.write(out_image)

            print(f"✅ 已裁切并保存: {output_raster_path}")

# 示例调用
clip_rasters_with_shapefile(
    raster_folder=r"D:\shanghai workshop\lst\original\clean\merge",
    shapefile_path=r"D:\shanghai workshop\vector\上海市_Polygon.shp",
    output_folder=r"D:\shanghai workshop\lst\original\clean\merge_clipped"
)

C:\Users\owner\AppData\Local\Temp\ipykernel_16756\37876436.py:28: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  polygon_union = polygon_proj.unary_union


✅ 已裁切并保存: D:\shanghai workshop\lst\original\clean\merge_clipped\clipped_LC08_merged_20150702_QA_PIXEL_clean.TIF


C:\Users\owner\AppData\Local\Temp\ipykernel_16756\37876436.py:28: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  polygon_union = polygon_proj.unary_union


✅ 已裁切并保存: D:\shanghai workshop\lst\original\clean\merge_clipped\clipped_LC08_merged_20150702_ST_B10.TIF
❌ 跳过（raster 不包含 polygon）: LC08_merged_20150709_QA_PIXEL_clean.TIF
❌ 跳过（raster 不包含 polygon）: LC08_merged_20150709_ST_B10.TIF


C:\Users\owner\AppData\Local\Temp\ipykernel_16756\37876436.py:28: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  polygon_union = polygon_proj.unary_union


...

✅ 已裁切并保存: D:\shanghai workshop\lst\original\clean\merge_clipped\clipped_LC08_merged_20240819_ST_B10.TIF
❌ 跳过（raster 不包含 polygon）: LC08_merged_20240826_QA_PIXEL_clean.TIF
❌ 跳过（raster 不包含 polygon）: LC08_merged_20240826_ST_B10.TIF


C:\Users\owner\AppData\Local\Temp\ipykernel_16756\37876436.py:28: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  polygon_union = polygon_proj.unary_union


✅ 已裁切并保存: D:\shanghai workshop\lst\original\clean\merge_clipped\clipped_LC08_merged_20240827_QA_PIXEL_clean.TIF


C:\Users\owner\AppData\Local\Temp\ipykernel_16756\37876436.py:28: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  polygon_union = polygon_proj.unary_union


✅ 已裁切并保存: D:\shanghai workshop\lst\original\clean\merge_clipped\clipped_LC08_merged_20240827_ST_B10.TIF


## 2 计算百分比

In [46]:
import rasterio
import numpy as np
import os
import re

def get_sorted_dates(file_path, suffix="QA_PIXEL_clean.TIF"):
    file_names = os.listdir(file_path)
    date_keys = []

    for fname in file_names:
        if fname.endswith(suffix):
            match = re.search(r'(\d{8})', fname)
            if match:
                date_keys.append(match.group(1))

    sorted_dates = sorted(set(date_keys))
    print("📅 提取的日期：", sorted_dates)
    return sorted_dates

def calculate_pixel_proportion(sorted_dates, base_folder, pixel_values=[21824, 21952]):
    for date_str in sorted_dates:
        # 找到匹配这个日期的文件（不拼死结构）
        matched_files = [f for f in os.listdir(base_folder)
                         if f.endswith("QA_PIXEL_clean.TIF") and date_str in f]

        if not matched_files:
            print(f"⚠️ 没有找到包含日期 {date_str} 的 QA 文件")
            continue

        file_path = os.path.join(base_folder, matched_files[0])
        print(f"📂 正在分析: {file_path}")

        with rasterio.open(file_path) as src:
            qa_pixel_data = src.read(1)

        non_zero_pixels = qa_pixel_data[qa_pixel_data > 0]
        total_non_zero_pixels = non_zero_pixels.size

        if total_non_zero_pixels == 0:
            print(f"⚠️ 所有像素为0: {file_path}")
            continue

        unique, counts = np.unique(non_zero_pixels, return_counts=True)
        pixel_counts = dict(zip(unique, counts))

        pixel_proportions = {val: pixel_counts.get(val, 0) / total_non_zero_pixels for val in pixel_values}

        print(f"📅 {date_str}:")
        for val in pixel_values:
            prop = pixel_proportions.get(val, 0)
            print(f"  - 像素值 {val} 的占比: {prop:.2%}")
        print()

# 示例路径
base_folder = os.path.join(work_space,'clean','merge_clipped')
sorted_dates = get_sorted_dates(base_folder)
calculate_pixel_proportion(sorted_dates, base_folder)

📅 提取的日期： ['20150702', '20150718', '20150803', '20150819', '20160602', '20160618', '20160704', '20160720', '20160805', '20160821', '20170605', '20170621', '20170707', '20170723', '20170808', '20170824', '20180608', '20180624', '20180710', '20180726', '20180811', '20180827', '20190611', '20190627', '20190713', '20190729', '20190814', '20190830', '20200613', '20200629', '20200715', '20200731', '20200816', '20210616', '20210702', '20210718', '20210803', '20210819', '20220603', '20220611', '20220619', '20220627', '20220705', '20220713', '20220721', '20220729', '20220806', '20220814', '20220822', '20220830', '20230606', '20230614', '20230622', '20230630', '20230708', '20230716', '20230724', '20230801', '20230809', '20230817', '20230825', '20240608', '20240616', '20240624', '20240702', '20240710', '20240718', '20240726', '20240803', '20240811', '20240819', '20240827']
📂 正在分析: D:\shanghai workshop\lst\original\clean\merge_clipped\clipped_LC08_merged_20150702_QA_PIXEL_clean.TIF
📅 20150702:
  - 

In [ ]:
import rasterio
import numpy as np

def find_high_proportion_dates(sorted_dates, input_folder, threshold=0.80, target_value=[21824,21952] ):
    high_proportion_dates = []

    for date in sorted_dates:
        # Path to the QA_PIXEL TIFF file for the current date
        data_path = rf'{input_folder}\clipped_LC08_merged_{date}_QA_PIXEL_clean.tif'

        try:
            with rasterio.open(data_path) as src:
                qa_pixel_data = src.read(1)  # Read the first band data

            # Filter for non-zero pixels
            non_zero_pixels = qa_pixel_data[qa_pixel_data > 0]
            total_non_zero_pixels = non_zero_pixels.size

            # Count unique pixel values and calculate proportions
            unique, counts = np.unique(non_zero_pixels, return_counts=True)
            pixel_counts = dict(zip(unique, counts))
            pixel_proportions = {value: count / total_non_zero_pixels for value, count in pixel_counts.items()}

            # Check the proportion of the target values
            proportion_target_value = sum(pixel_proportions.get(val, 0) for val in target_value)
            if proportion_target_value > threshold:
                high_proportion_dates.append((date, proportion_target_value))
                print(f"{date} pixel values {target_value} combined proportion: {proportion_target_value:.2%}")

        except FileNotFoundError:
            print(f"File not found: {data_path}")

    return high_proportion_dates

input_folder= os.path.join(work_space,'clean','merge_clipped')
high_proportion_dates = find_high_proportion_dates(sorted_dates, input_folder)

# Print results
print("\nDates with high target value proportion:")
for date, proportion in high_proportion_dates:
    print(f"{date}: {proportion:.2%}")

20150803 pixel values [21824, 21952] combined proportion: 97.98%
20150819 pixel values [21824, 21952] combined proportion: 83.20%


# 用 QA数据清理B10 -> clean data

In [17]:
import rasterio
import numpy as np
import os
for date, proportion in high_proportion_dates:
    data_workspace = os.path.join(r"clean\merge_clipped")
    b_10_data = os.path.join(data_workspace, rf'clipped_{date}_ST_B10.tif')
    qa_data = os.path.join(data_workspace, rf'clipped_{date}_QA_PIXEL.tif')
    output_folder = os.path.join(data_workspace,r'clean')
    os.makedirs(output_folder,exist_ok=True)

    output_data = os.path.join(output_folder, rf'clipped_{date}_B10_cleaned.tif')
    print("B10 Data Path:", b_10_data)
    print("QA Data Path:", qa_data)
    print("Output Data Path:", output_data)

    # Read Band 10 and QA data
    with rasterio.open(b_10_data) as src:
        band10 = src.read(1).astype(float)  # Convert data to float type to support NaN
        profile = src.profile

    with rasterio.open(qa_data) as qa_src:
        qa = qa_src.read(1)  # Read QA pixel data

    # Define labels for different cloud and shadow conditions
    label = [1, 21824, 21826, 21888, 21890, 22080, 22144, 22280, 23888, 23952, 24088, 24216, 24344, 24472, 30048, 54596, 54852, 55052]
    print("Dilated cloud over land QA label:", label[2])

    # Create a mask for clouds and shadows, filtering out cloud and shadow areas
    valid_labels = [21824, 21952]
    cloud_mask = ~np.isin(qa, valid_labels)  # mask 出非有效值（云、阴影、水等）

    # Set cloud and shadow areas to 0
    band10[cloud_mask] = 0

    # Save the processed result
    with rasterio.open(output_data, 'w', **profile) as dst:
        dst.write(band10, 1)


B10 Data Path: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_ST_B10.tif
QA Data Path: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_QA_PIXEL.tif
Output Data Path: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clean\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_B10_cleaned.tif
Dilated cloud over land QA label: 21826
B10 Data Path: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clipped_LC08_L2SP_116034_20200919_20201006_02_T1_ST_B10.tif
QA Data Path: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clipped_LC08_L2SP_116034_20200919_20201006_02_T1_QA_PIXEL.tif
Output Data Path: D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clean\clipped_LC08_L2SP_116034_20200919_20201006_02_T1_B10_cleaned.tif
Dilated cloud over land QA label: 21826


# 先将数据洗成摄氏度 bt -> lst

In [6]:
import rasterio
import numpy as np
import os

def calculate_lst(input_files, output_dir, multiplier=0.00341802, add_constant=149.0, nodata_value=-9999):
    """
    Calculate Land Surface Temperature (LST) from a list of raster files and save the output.

    Args:
        input_files (list): List of paths to input raster files.
        output_dir (str): Directory to save the output raster files.
        multiplier (float): Multiplication factor for converting to temperature.
        add_constant (float): Additive constant for converting to temperature.
        nodata_value (float): Value to use for no-data in the output files.
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    for input_file in input_files:
        with rasterio.open(input_file) as src:
            # Read the first band and convert to float
            band_data = src.read(1).astype(float)
            profile = src.profile

        # Replace values <= 0 with NaN
        band_data[band_data <= 0] = np.nan

        # Calculate brightness temperature (BT)
        lst_kelvin = (band_data * multiplier) + add_constant

        # Convert to Celsius
        lst_celsius = lst_kelvin - 273.15

        # Replace NaN values with the nodata value
        lst_celsius[np.isnan(lst_celsius)] = nodata_value

        # Update profile for the output file
        profile.update({
            'dtype': 'float32',
            'count': 1,
            'nodata': nodata_value
        })

        # Generate a short output file name
        base_name = os.path.basename(input_file).split('.')[0]
        output_file = os.path.join(output_dir, f"{base_name}_LST.tif")

        # Save the LST raster
        with rasterio.open(output_file, 'w', **profile) as dst:
            dst.write(lst_celsius, 1)

        print(f"LST saved to: {output_file}")


# 输入的栅格文件路径
normal_files = [
    r'D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clean\clipped_LC08_L2SP_116034_20200919_20201006_02_T1_B10_cleaned.tif'
]
extreme_files = [
    r'D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\clean\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_B10_cleaned.tif'
]

# 输出目录
output_dir_normal = r'D:\seoul\Final_data\landsat_final\data\normal_heat_day_lst'
output_dir_extreme = r'D:\seoul\Final_data\landsat_final\data\extreme_heat_day_lst'

# 计算并保存 LST
calculate_lst(normal_files, output_dir_normal)
calculate_lst(extreme_files, output_dir_extreme)

LST saved to: D:\seoul\Final_data\landsat_final\data\normal_heat_day_lst\clipped_LC08_L2SP_116034_20200919_20201006_02_T1_B10_cleaned_LST.tif
LST saved to: D:\seoul\Final_data\landsat_final\data\extreme_heat_day_lst\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_B10_cleaned_LST.tif


# 相减，然后得到HR map

In [7]:
import rasterio
import numpy as np
import os

def calculate_heat_resilience(normal_files, extreme_files, output_dir, nodata_value=-9999):
    os.makedirs(output_dir, exist_ok=True)

    if len(normal_files) != len(extreme_files):
        raise ValueError("Number of normal and extreme files must be the same.")

    for normal_file, extreme_file in zip(normal_files, extreme_files):
        with rasterio.open(normal_file) as normal_src, rasterio.open(extreme_file) as extreme_src:
            normal_data = normal_src.read(1).astype(float)
            extreme_data = extreme_src.read(1).astype(float)

            if normal_data.shape != extreme_data.shape:
                raise ValueError(f"Dimension mismatch between {normal_file} and {extreme_file}.")

            # 强制处理 nodata
            if normal_src.nodata is not None:
                normal_data[normal_data == normal_src.nodata] = np.nan
            else:
                normal_data[normal_data == nodata_value] = np.nan

            if extreme_src.nodata is not None:
                extreme_data[extreme_data == extreme_src.nodata] = np.nan
            else:
                extreme_data[extreme_data == nodata_value] = np.nan

            # 差值
            heat_resilience = normal_data - extreme_data

            # 打印调试
            print(f"\n📁 Processing: {os.path.basename(normal_file)} vs {os.path.basename(extreme_file)}")
            print("🔹 First 10 values from normal_data (non-NaN):", normal_data[~np.isnan(normal_data)].flatten()[:10])
            print("🔹 First 10 values from extreme_data (non-NaN):", extreme_data[~np.isnan(extreme_data)].flatten()[:10])
            print("🔸 First 10 values from heat_resilience (non-NaN):", heat_resilience[~np.isnan(heat_resilience)].flatten()[:10])

            # 替换 NaN 为 nodata
            heat_resilience[np.isnan(heat_resilience)] = nodata_value

            profile = normal_src.profile
            profile.update({
                'dtype': 'float32',
                'nodata': nodata_value
            })

            year = os.path.basename(normal_file).split('_')[4][:4]
            output_file = os.path.join(output_dir, f"{year}_heat_resilience.tif")

            with rasterio.open(output_file, 'w', **profile) as dst:
                dst.write(heat_resilience.astype('float32'), 1)

            print(f"✅ Heat resilience raster saved to: {output_file}")




# 输入的栅格文件路径
normal_files = [
    r'D:\seoul\Final_data\landsat_final\data\normal_heat_day_lst\clipped_LC08_L2SP_116034_20200919_20201006_02_T1_B10_cleaned_LST.tif'
]
extreme_files = [
    r'D:\seoul\Final_data\landsat_final\data\extreme_heat_day_lst\clipped_LC08_L2SP_116034_20200530_20200820_02_T1_B10_cleaned_LST.tif'
]

# 输出目录
output_dir = r'D:\seoul\Final_data\landsat_final\data\heat_resilience'

# 计算并保存 heat resilience
calculate_heat_resilience(normal_files, extreme_files, output_dir, nodata_value=-9999)


📁 Processing: clipped_LC08_L2SP_116034_20200919_20201006_02_T1_B10_cleaned_LST.tif vs clipped_LC08_L2SP_116034_20200530_20200820_02_T1_B10_cleaned_LST.tif
🔹 First 10 values from normal_data (non-NaN): [22.07631302 22.79067993 22.37709999 22.72915459 22.42153358 23.30680084
 23.11539078 23.31705475 23.60074997 23.68278313]
🔹 First 10 values from extreme_data (non-NaN): [26.02754402 26.52999306 25.89424133 26.16084671 26.15401077 26.85128784
 26.57101059 26.64962387 26.75558281 26.70431328]
🔸 First 10 values from heat_resilience (non-NaN): [-3.951231   -3.73931313 -3.51714134 -3.43169212 -3.73247719 -3.544487
 -3.45561981 -3.33256912 -3.15483284 -3.02153015]
✅ Heat resilience raster saved to: D:\seoul\Final_data\landsat_final\data\heat_resilience\2020_heat_resilience.tif


# 计算面积和占比

In [3]:
# import geopandas as gpd
# import os
# import pandas as pd
# from concurrent.futures import ThreadPoolExecutor
# def add_lst_ratio_within_only(clipped_gdf, city_path, output_path):
#     # ✅ STEP 1: 读取 grid 文件（即 city_gdf），用其 CRS 作为标准
#     city_gdf = gpd.read_file(city_path)
#     target_crs = city_gdf.crs  # ⬅️ 使用它为主
#     clipped = clipped_gdf.to_crs(target_crs)
#     city_gdf = city_gdf.to_crs(target_crs)
#
#     # ✅ STEP 2: 创建 city_id 和 Shape_Area
#     city_gdf = city_gdf.reset_index(drop=True)
#     city_gdf['city_id'] = city_gdf.index
#     # city_gdf['Shape_Area'] = city_gdf.geometry.area
#     city_gdf['lst'] = 0.0  # 初始化
#
#     # ✅ STEP 3: 找出每个 grid 内完全包含的 clipped polygon
#     for idx, row in city_gdf.iterrows():
#         grid_geom = row.geometry
#         inside = clipped[clipped.within(grid_geom)]
#         if not inside.empty:
#             area_sum = inside.geometry.area.sum()
#             city_gdf.at[idx, 'lst'] = area_sum
#
#     # ✅ STEP 4: 计算 lst_ratio
#     city_gdf['lst_ratio'] = city_gdf['lst'] / city_gdf['Shape_Area']
#
#     # ✅ STEP 5: 导出为 ITRF_2000_UTM_K
#     itrf_proj = (
#         '+proj=tmerc +lat_0=38 +lon_0=127.5 '
#         '+k=0.9996 +x_0=1000000 +y_0=2000000 '
#         '+ellps=GRS80 +units=m +no_defs'
#     )
#     city_gdf_final = city_gdf.to_crs(itrf_proj)
#     city_gdf_final.to_file(output_path, driver='ESRI Shapefile')
#
#     print(f"✅ 完全包含统计完成并保存: {output_path}")
#
#
#
#
# # === 设置路径 ===
# # 所有 clipped shapefiles（一个 per grid）
# clipped_folder = r'D:\seoul\Final_data\landsat_final\data\landsat8_data_merge_clipped\shapefiles_clipped_with_fid'
# clipped_files = [f for f in os.listdir(clipped_folder) if f.endswith('.shp')]
#
# # 所有 grid shapefiles
# grid_folder = r'D:\seoul\grids'
# grid_files = [f for f in os.listdir(grid_folder) if f.endswith('.shp')]
#
# # 输出文件夹
# output_folder = r'D:\seoul\grids\city_ratio_outputs'
# os.makedirs(output_folder, exist_ok=True)
#
# def process_pair(grid_file):
#     try:
#         grid_name = os.path.splitext(grid_file)[0]
#         grid_path = os.path.join(grid_folder, grid_file)
#
#         matched_clipped = [
#             f for f in clipped_files if grid_name in f
#         ]
#
#         if not matched_clipped:
#             print(f"⚠️ 找不到与 {grid_file} 匹配的 clipped 文件")
#             return
#
#         clipped_path = os.path.join(clipped_folder, matched_clipped[0])
#         clipped_gdf = gpd.read_file(clipped_path)
#         clipped_gdf = clipped_gdf.reset_index(drop=True)
#         clipped_gdf['city_id'] = clipped_gdf['FID'] if 'FID' in clipped_gdf.columns else clipped_gdf.index
#
#         output_path = os.path.join(output_folder, f'city2020_lst_ratio_{grid_name}.shp')
#         if os.path.exists(output_path):
#             print(f"⏩ 已存在，跳过: {output_path}")
#             return
#
#         add_lst_ratio_within_only(clipped_gdf, grid_path, output_path)
#
#     except Exception as e:
#         print(f"❌ 处理 {grid_file} 失败: {e}")
#
# # 🚀 并行处理
# print("🚀 开始批量匹配 grid ↔ clipped...")
# with ThreadPoolExecutor(max_workers=4) as executor:
#     executor.map(process_pair, grid_files)

🚀 开始批量匹配 grid ↔ clipped...
⏩ 已存在，跳过: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_1080m.shp
⏩ 已存在，跳过: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_360m.shp
⏩ 已存在，跳过: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_240m.shp
⏩ 已存在，跳过: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_600m.shp
⏩ 已存在，跳过: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_480m.shp
⏩ 已存在，跳过: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_120m.shp
⏩ 已存在，跳过: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_840m.shp
⏩ 已存在，跳过: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_960m.shp
⏩ 已存在，跳过: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_720m.shp
✅ 完全包含统计完成并保存: D:\seoul\grids\city_ratio_outputs\city2020_lst_ratio_grid_450m.shp
